In [1]:
#  LOAD THE CLEANED DATASET
# Import pandas for data handling
import pandas as pd

# Define the data folder path
path = r'C:\Users\pranv\OneDrive\Desktop\Hospital Operations & Patient Analytics\Hospital-Operations-Patient-Analytics-\data\\'

# Load Module 2's output dataset 
df_cleaned = pd.read_csv(path + 'hospital_cleaned.csv')

# Work on a COPY of the data
df = df_cleaned.copy()

# Display 
print("Shape of the dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
df.head()

Shape of the dataset: (55500, 45)

Column names:
['Patient_Name', 'Age', 'Gender', 'Blood Type', 'Diagnosis', 'Admission_Date', 'Doctor', 'Hospital_Name', 'Insurance', 'Billing Amount', 'Room_No', 'Admission_Type', 'Discharge_Date', 'Medication', 'Test_Result', 'Department', 'Hospital_ID', 'Hospital_Type', 'City', 'State', 'Department_ID', 'Nurses', 'Dept_Staff_Count', 'Dept_Beds', 'Dept_ICU_Beds', 'Patient_ID', 'Bed_Number', 'ICU_Beds', 'Beds_Available', 'Bed_Occupied', 'Equipment_ID', 'Equipment_Type', 'Equipment_Status', 'Equipment_InUse_Flag', 'Length_of_Stay', 'Transferred', 'Number_of_Transfers', 'Readmission', 'Staff_Utilization', 'Bed_Occupancy_Rate', 'ICU_Occupancy_Rate', 'Bed_Occupancy_Rate_Normalized', 'ICU_Occupancy_Rate_Normalized', 'Staff_Utilization_Normalized', 'Length_of_Stay_Normalized']


,Patient_Name,Age,Gender,Blood Type,Diagnosis,Admission_Date,Doctor,Hospital_Name,Insurance,Billing Amount,...,Transferred,Number_of_Transfers,Readmission,Staff_Utilization,Bed_Occupancy_Rate,ICU_Occupancy_Rate,Bed_Occupancy_Rate_Normalized,ICU_Occupancy_Rate_Normalized,Staff_Utilization_Normalized,Length_of_Stay_Normalized
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281306,...,No,0,Yes,86.6,109.5,85.0,0.792904,0.928571,1.000000,0.034483
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,...,Yes,1,No,63.7,35.9,44.3,0.259957,0.347143,0.139098,0.172414
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096079,...,No,0,No,63.7,32.8,51.8,0.237509,0.454286,0.139098,0.482759
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.782410,...,No,0,No,71.3,19.4,24.6,0.140478,0.065714,0.424812,1.000000
4,adrIENNE bEll,43,Female,Ab+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,...,No,0,No,86.6,0.0,58.7,0.000000,0.552857,1.000000,0.655172


In [2]:
#  CALCULATE TOTAL ADMISSIONS


# Total Admissions = count of unique patient records in the dataset
total_admissions = df['Patient_ID'].nunique()
print("Total Admissions:", total_admissions)

# Add as a new column — same overall value repeated for every row

df['Total_Admissions'] = total_admissions

Total Admissions: 55500


In [3]:
# CALCULATE OCCUPANCY RATE


# Occupancy Rate = (Occupied beds / Total department beds) * 100
# Count how many patient records have their bed marked as occupied
occupied_beds = (df['Bed_Occupied'] == 'Yes').sum()

# Sum up total beds across all department records
total_dept_beds = df['Dept_Beds'].sum()

# Calculate the overall occupancy rate as a percentage
overall_occupancy_rate = round((occupied_beds / total_dept_beds) * 100, 2)
print("Overall Occupancy Rate:", overall_occupancy_rate)

# Add as a new column — same overall value repeated for every row
df['Overall_Occupancy_Rate'] = overall_occupancy_rate

Overall Occupancy Rate: 1.17


In [4]:
#  CALCULATE AVERAGE LENGTH OF STAY

# Average Length of Stay = mean of the Length_of_Stay column (in days)
avg_length_of_stay = round(df['Length_of_Stay'].mean(), 2)
print("Average Length of Stay (days):", avg_length_of_stay)

# Add as a new column  
df['Average_Length_of_Stay'] = avg_length_of_stay

Average Length of Stay (days): 15.51


In [5]:
#  CALCULATE READMISSION RATE

# Readmission Rate = (Patients readmitted / Total patients) * 100
readmitted_count = (df['Readmission'] == 'Yes').sum()
total_patients = len(df)

readmission_rate = round((readmitted_count / total_patients) * 100, 2)
print("Readmission Rate (%):", readmission_rate)

# Add as a new column 
df['Readmission_Rate'] = readmission_rate

Readmission Rate (%): 10.05


In [6]:
#  CALCULATE BED UTILIZATION RATE

# Bed Utilization Rate = (Occupied beds in department / Total department beds) * 100

df['Bed_Utilization_Rate'] = (
    (df['Dept_Beds'] - df['Beds_Available']) / df['Dept_Beds'] * 100
).round(2)

# Preview to confirm it varies sensibly by department
df[['Department', 'Dept_Beds', 'Beds_Available', 'Bed_Utilization_Rate']].head(10)

,Department,Dept_Beds,Beds_Available,Bed_Utilization_Rate
0,Oncology,21,-2,109.52
1,General Medicine,64,41,35.94
2,General Medicine,64,43,32.81
3,Endocrinology,67,54,19.40
4,Oncology,21,21,0.00
5,Pulmonology,81,65,19.75
6,Endocrinology,67,47,29.85
7,Oncology,21,14,33.33
8,Pulmonology,81,66,18.52
9,Oncology,21,1,95.24


In [7]:
# CALCULATE DEPARTMENT EFFICIENCY SCORE


# Department Efficiency Score
# 1. Staff Utilization (40% weight) - how efficiently staff are being used
# 2. Bed Occupancy Rate (40% weight) - how well beds are being utilized
# 3. Readmission penalty (20% weight) - lower readmissions = higher efficiency

# Formula: weighted sum of normalised (0-1 scale) components
df['Department_Efficiency_Score'] = (
    (df['Staff_Utilization'] / 100) * 0.4 +
    (df['Bed_Occupancy_Rate'] / 100) * 0.4 +
    (1 - (df['Readmission'] == 'Yes').astype(int)) * 0.2
).round(4)

# Preview the result
df[['Department', 'Staff_Utilization', 'Bed_Occupancy_Rate', 
    'Readmission', 'Department_Efficiency_Score']].head(10)

,Department,Staff_Utilization,Bed_Occupancy_Rate,Readmission,Department_Efficiency_Score
0,Oncology,86.6,109.5,Yes,0.7844
1,General Medicine,63.7,35.9,No,0.5984
2,General Medicine,63.7,32.8,No,0.5860
3,Endocrinology,71.3,19.4,No,0.5628
4,Oncology,86.6,0.0,No,0.5464
5,Pulmonology,60.0,19.8,No,0.5192
6,Endocrinology,71.3,29.9,No,0.6048
7,Oncology,86.6,33.3,No,0.6796
8,Pulmonology,60.0,18.5,No,0.5140
9,Oncology,86.6,95.2,No,0.9272


In [8]:

# Print final shape to confirm all new KPI columns were added
print("Final dataset shape:", df.shape)

# Print a summary of all calculated KPIs for a final check
print("\nKPI Summary:")
print(df[['Total_Admissions', 'Overall_Occupancy_Rate', 'Average_Length_of_Stay',
           'Readmission_Rate', 'Bed_Utilization_Rate', 'Department_Efficiency_Score']].describe())

# Save the final dataset 
save_path = path + 'hospital_final_dataset.xlsx'
df.to_excel(save_path, index=False, engine='openpyxl')

print("\nhospital_final_dataset.xlsx saved successfully.")
print("Saved to:", save_path)

Final dataset shape: (55500, 51)

KPI Summary:
       Total_Admissions  Overall_Occupancy_Rate  Average_Length_of_Stay  \
count           55500.0            5.550000e+04            5.550000e+04   
mean            55500.0            1.170000e+00            1.551000e+01   
std                 0.0            1.288536e-12            7.547808e-12   
min             55500.0            1.170000e+00            1.551000e+01   
25%             55500.0            1.170000e+00            1.551000e+01   
50%             55500.0            1.170000e+00            1.551000e+01   
75%             55500.0            1.170000e+00            1.551000e+01   
max             55500.0            1.170000e+00            1.551000e+01   

       Readmission_Rate  Bed_Utilization_Rate  Department_Efficiency_Score  
count      5.550000e+04          55500.000000                 55500.000000  
mean       1.005000e+01             29.598808                     0.580502  
std        4.927658e-12             26.938455 

In [10]:
print(df['Bed_Utilization_Rate'].describe())

count    55500.000000
mean        29.598808
std         26.938455
min          0.000000
25%         11.940000
50%         24.240000
75%         37.310000
max        138.100000
Name: Bed_Utilization_Rate, dtype: float64


In [11]:
# Fix: Bed_Utilization_Rate should never exceed 100% or go below 0%
df['Bed_Utilization_Rate'] = df['Bed_Utilization_Rate'].clip(lower=0, upper=100)

print("Bed_Utilization_Rate range after fix:")
print(df['Bed_Utilization_Rate'].describe())

Bed_Utilization_Rate range after fix:
count    55500.000000
mean        28.700562
std         23.967468
min          0.000000
25%         11.940000
50%         24.240000
75%         37.310000
max        100.000000
Name: Bed_Utilization_Rate, dtype: float64


In [12]:
save_path = path + 'hospital_final_dataset.xlsx'
df.to_excel(save_path, index=False, engine='openpyxl')

import os
print("Re-saved with fix. File size (bytes):", os.path.getsize(save_path))

Re-saved with fix. File size (bytes): 17470004


In [13]:
# Fix: Department_Efficiency_Score should stay within 0 to 1 range
df['Department_Efficiency_Score'] = df['Department_Efficiency_Score'].clip(lower=0, upper=1)

print("Department_Efficiency_Score range after fix:")
print(df['Department_Efficiency_Score'].describe())

Department_Efficiency_Score range after fix:
count    55500.000000
mean         0.579063
std          0.139879
min          0.240000
25%          0.503200
50%          0.561200
75%          0.623600
max          1.000000
Name: Department_Efficiency_Score, dtype: float64


In [14]:
save_path = path + 'hospital_final_dataset.xlsx'
df.to_excel(save_path, index=False, engine='openpyxl')

import os
print("Re-saved. File size (bytes):", os.path.getsize(save_path))

Re-saved. File size (bytes): 17466573
